# Probing a Model

This notebook shows how to evaluate a model's embeddings with **linear probing**.
After generating embeddings with a pretrained backbone, we train a small linear
classifier (a *probe*) on top of the *frozen* embeddings to measure how well your
own ground-truth species can be linearly separated. We then load the trained
probe and apply it to new embeddings to generate predictions.


Make sure to reset your config and settings files before running this notebook, 
as they may contain settings from previous runs that could prevent this notebook 
from finding the correct paths.

---
## 1. Setup & Configuration
Import necessary modules 

In [24]:
# to run successfully the packages for jupyter notebook need to be installed:
# uv pip install ipykernel, ipython

from IPython.display import display
import os 
from pathlib import Path

# load the specific package
import bacpipe
import classifier

Set the working directory to the repository root and clean the previous tests if needed/wanted

In [25]:
bacpipe.config.audio_dir = r"C:\Users\Conservación Osa 9\Desktop\Silencio"
bacpipe.settings.main_results_dir = 'bacpipe_results'
bacpipe.config.already_computed=True

In [26]:
import importlib.resources as pkg_resources
os.chdir(pkg_resources.files("bacpipe"))
os.chdir('..')
print(os.listdir('.'))


# Change the value of the key main_results_dir in the namespace bacpipe.settings to change the directory 
# where the results of the tutorials are stored. By default, it is set to './bacpipe_results'.
bacpipe.settings.main_results_dir = str(Path(bacpipe.settings.main_results_dir) / 'probing_a_model')

# !WARNING! the following code deletes the folder where the results of this tutorial is stored to be sure to start with a clean folder. 
# If you have important data in this folder, please comment it before running this code.
folder_path = bacpipe.settings.main_results_dir
if os.path.exists(folder_path):
    # Prompt the user
    user_input = input(f"Are you sure you want to delete '{folder_path}'? (y/n): ").lower().strip()

    if user_input == 'y':
        import shutil
        shutil.rmtree(folder_path) 
        print(f"Folder {folder_path} deleted.")
    else:
        print("Operation cancelled.")

else:
    print(f"Folder {folder_path} not found.")

['.git', '.gitattributes', '.github', '.gitignore', '.pre-commit-config.yaml', '.pytest_cache', '.python-version', '.readthedocs.yml', '.venv', 'bacpipe', 'bacpipe_model_checkpoints', 'bacpipe_results', 'classifier', 'docs', 'hierachry.json', 'LICENSE', 'probing_a_model.ipynb', 'probing_hierachy.ipynb', 'probing_PU.ipynb', 'pyproject.toml', 'README.md', 'requirements_no_cuda.txt', 'requirements_no_tf.txt', 'requirements_tf_gpu.txt', 'run_pipeline.py', 'Silencio_H23', 'src']
Operation cancelled.


Set the global constants used throughout the notebook.

In [27]:
print(os.listdir("."))


['.git', '.gitattributes', '.github', '.gitignore', '.pre-commit-config.yaml', '.pytest_cache', '.python-version', '.readthedocs.yml', '.venv', 'bacpipe', 'bacpipe_model_checkpoints', 'bacpipe_results', 'classifier', 'docs', 'hierachry.json', 'LICENSE', 'probing_a_model.ipynb', 'probing_hierachy.ipynb', 'probing_PU.ipynb', 'pyproject.toml', 'README.md', 'requirements_no_cuda.txt', 'requirements_no_tf.txt', 'requirements_tf_gpu.txt', 'run_pipeline.py', 'Silencio_H23', 'src']


In [28]:
MODEL_NAME = 'aves_especies'                  # name of the model to run. Supported models are in bacpipe.supported_models
AUDIO_DIR = r"C:\Users\Conservación Osa 9\Desktop\Silencio"   # path to directory containing audio files

In [29]:
bacpipe.settings.train_ratio = 0.75
bacpipe.settings.test_ratio = 0.2

In [30]:
species_label_map = {
    'DENMIC' : "microcephalus",
    'DENEBR' : "ebraccatus", 
    'ENGPUS' : "pustulosus", 
    'LEPFRA' : "fragilis",
    'ELAPER' : "pearsei"
}

In [31]:
embeds = bacpipe.run_pipeline_for_single_model(
    model_name=MODEL_NAME,
    audio_dir=AUDIO_DIR,
    check_if_already_processed=True,
    only_embed_annotations=False,
).embeddings(return_type='array')

gt = bacpipe.ground_truth_by_model(
    model=MODEL_NAME,
    audio_dir=AUDIO_DIR,
    annotations_filename='annotations.csv',
    overwrite=False,
    only_embed_annotations=False,
)


Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

aves_especies checkpoint exists.




###### Generating embeddings using AVES_ESPECIES ######

Finding all generated embeddings: 104it [00:00, 9716.82it/s]
Found 104 embedding files.
finding audio files: 113it [00:00, 21277.50it/s]
Found 104 number of audio files.

### Embeddings already exist. Using embeddings in bacpipe_results/probing_a_model/Silencio/embeddings/2026-11-09_09-30___aves_especies-Silencio ###


In [34]:
import random
import numpy as np
import pandas as pd
import torch

mixup_alphas = [round(value / 10, 1) for value in range(1, 11)]
mixup_probabilities = [0.0, 0.5, 1.0]
comparison_rows = []

for alpha in mixup_alphas:
    for probability in mixup_probabilities:
        seed = int(round(alpha * 100)) * 100 + int(round(probability * 10))
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        print(
            "Running linear probe: "
            f"mixup_alpha={alpha}, mixup_probability={probability}"
        )
        probe, label2idx, metrics = classifier.probing_pipeline(
            model_name=MODEL_NAME,
            ground_truth=gt,
            embeds=embeds,
            overwrite=True,
            only_embed_annotations=False,
            name="hierarchy",
            manifold_mixup=True,
            manifold_mixup_alpha=alpha,
            manifold_mixup_probability=probability,
            species_label_map=species_label_map
        )

        row = {
            "mixup_alpha": alpha,
            "mixup_probability": probability,
        }
        row.update(metrics.get("overall", {}))
        comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values(
    ["mixup_alpha", "mixup_probability"]
).reset_index(drop=True)

results_dir = Path(bacpipe.settings.main_results_dir)
results_dir.mkdir(parents=True, exist_ok=True)
comparison_path = results_dir / "mixup_linear_probe_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print(f"Comparison saved to {comparison_path}")



Found file: bacpipe_results\probing_a_model\Silencio\evaluations\aves_especies\labels\probing_dataframe.csv. Loading dataframe probing dataframe. If you would like to automatically create a new probing dataframe. Please delete the existing one.



Running linear probe: mixup_alpha=0.1, mixup_probability=0.0


getting time of day: 100%|██████████| 104/104 [00:00<?, ?it/s]
getting time per embeddings: 104it [00:00, 5933.83it/s]
getting day of year: 100%|██████████| 104/104 [00:00<?, ?it/s]
getting continuous timestamps: 104it [00:00, 5620.15it/s]
getting parent directory: 104it [00:00, ?it/s]
getting audio file names: 104it [00:00, 34702.28it/s]
Building metadata labels: 100%|██████████| 6/6 [00:00<00:00, 73.75it/s]

You have passed a multi-label ground truth array. However for visualization only one label will be displayed.

c:\bacpipe\bacpipe\embedding_evaluation\visualization\visualize_embeddings.py:465: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ground_truth['start'] = [np.round(v, 4) for v in df_ground_truth['start']]
Found 2332 samples in th

Running linear probe: mixup_alpha=0.1, mixup_probability=0.5


getting time of day: 100%|██████████| 104/104 [00:00<?, ?it/s]
getting time per embeddings: 104it [00:00, 6606.60it/s]
getting day of year: 100%|██████████| 104/104 [00:00<?, ?it/s]
getting continuous timestamps: 104it [00:00, 4728.03it/s]
getting parent directory: 104it [00:00, ?it/s]
getting audio file names: 104it [00:00, ?it/s]
Building metadata labels: 100%|██████████| 6/6 [00:00<00:00, 98.89it/s]

You have passed a multi-label ground truth array. However for visualization only one label will be displayed.

c:\bacpipe\bacpipe\embedding_evaluation\visualization\visualize_embeddings.py:465: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ground_truth['start'] = [np.round(v, 4) for v in df_ground_truth['start']]
Found 2332 samples in the train

Running linear probe: mixup_alpha=0.1, mixup_probability=1.0


... (52 outputs left out)

getting time of day: 100%|██████████| 104/104 [00:00<?, ?it/s]
getting time per embeddings: 104it [00:00, 4230.51it/s]
getting day of year: 100%|██████████| 104/104 [00:00<?, ?it/s]
getting continuous timestamps: 104it [00:00, 1273.90it/s]
getting parent directory: 104it [00:00, 16866.09it/s]<00:00, 25.24it/s]
getting audio file names: 104it [00:00, ?it/s]
Building metadata labels: 100%|██████████| 6/6 [00:00<00:00, 33.95it/s]

You have passed a multi-label ground truth array. However for visualization only one label will be displayed.

c:\bacpipe\bacpipe\embedding_evaluation\visualization\visualize_embeddings.py:465: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ground_truth['start'] = [np.round(v, 4) for v in df_ground_truth['start']]
Found 

Running linear probe: mixup_alpha=1.0, mixup_probability=1.0


getting time of day: 100%|██████████| 104/104 [00:00<?, ?it/s]
getting time per embeddings: 104it [00:00, 3845.51it/s]
getting day of year: 100%|██████████| 104/104 [00:00<00:00, 104281.05it/s]
getting continuous timestamps: 104it [00:00, 1733.41it/s]
getting parent directory: 104it [00:00, 21199.83it/s]<00:00, 27.46it/s]
getting audio file names: 104it [00:00, 25769.93it/s]
Building metadata labels: 100%|██████████| 6/6 [00:00<00:00, 36.14it/s]

You have passed a multi-label ground truth array. However for visualization only one label will be displayed.

c:\bacpipe\bacpipe\embedding_evaluation\visualization\visualize_embeddings.py:465: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ground_truth['start'] = [np.round(v, 4) for v in df_ground_tru

Comparison saved to bacpipe_results\probing_a_model\mixup_linear_probe_comparison.csv


In [35]:
display(comparison_df)

import matplotlib.pyplot as plt

metric_columns = [
    column for column in comparison_df.columns
    if column not in {"mixup_alpha", "mixup_probability"}
]
num_metrics = len(metric_columns)
fig, axes = plt.subplots(
    nrows=(num_metrics + 1) // 2,
    ncols=2,
    figsize=(14, 4 * ((num_metrics + 1) // 2)),
    squeeze=False,
)

for axis, metric in zip(axes.flat, metric_columns):
    pivot = comparison_df.pivot(
        index="mixup_alpha",
        columns="mixup_probability",
        values=metric,
    )
    pivot.plot(ax=axis, marker="o")
    axis.set_title(metric.replace("_", " ").title())
    axis.set_xlabel("Mixup alpha")
    axis.set_ylabel(metric.replace("_", " ").title())
    axis.grid(alpha=0.3)
    axis.legend(
        title="Mixup probability",
        labels=[str(value) for value in pivot.columns],
    )

for axis in axes.flat[num_metrics:]:
    axis.set_visible(False)

fig.suptitle("Linear Probe Mixup Comparison", fontsize=16)
fig.tight_layout()

comparison_figure_path = results_dir / "mixup_linear_probe_comparison.png"
fig.savefig(comparison_figure_path, dpi=150, bbox_inches="tight")
print(f"Figure saved to {comparison_figure_path}")
plt.show()


,mixup_alpha,mixup_probability,macro_accuracy,auc,macro_f1
0,0.1,0.0,0.861660,0.997344,0.900622
1,0.1,0.5,0.860780,0.997222,0.894164
2,0.1,1.0,0.849135,0.996926,0.889288
3,0.2,0.0,0.856252,0.997490,0.895269
4,0.2,0.5,0.827243,0.996938,0.865071
5,0.2,1.0,0.813479,0.996786,0.861629
6,0.3,0.0,0.852161,0.997630,0.889708
7,0.3,0.5,0.834594,0.997117,0.869574
8,0.3,1.0,0.843533,0.997028,0.874531
9,0.4,0.0,0.848837,0.997340,0.888736


Figure saved to bacpipe_results\probing_a_model\mixup_linear_probe_comparison.png


C:\Users\Conservación Osa 9\AppData\Local\Temp\ipykernel_6464\406902329.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 3. Probe Inference

`bacpipe.prepare_probe_inference` loads a probe that was trained and saved by the
probing pipeline, together with the `label2index.json` file describing the
class-to-column mapping. The returned probe is restored to its exact post-training
state and is ready for inference.

`bacpipe.run_probe_inference` applies that probe to embeddings and returns the
predictions. With `return_binary_presence=False` it returns the softmax
probabilities for each class; with `return_binary_presence=True` (the default) it
thresholds the probabilities with `threshold` (default `0.5`) and returns a binary
presence matrix.


In [9]:
probe, label2idx = bacpipe.prepare_probe_inference(model=MODEL_NAME)

predictions = bacpipe.run_probe_inference(
    model=MODEL_NAME,
    linear_probe=probe,
    threshold=0.5,
    embeds=embeds,
    return_binary_presence=False
)

display(predictions)


array([[0.93027884, 0.06972119],
       [0.89899176, 0.10100827],
       [0.940024  , 0.05997603],
       [0.93970525, 0.06029478],
       [0.9343625 , 0.06563759],
       [0.9287107 , 0.07128935],
       [0.9457818 , 0.05421818],
       [0.9384681 , 0.06153192],
       [0.9276346 , 0.0723654 ],
       [0.93313265, 0.06686729],
       [0.8974573 , 0.10254265],
       [0.9380407 , 0.06195933],
       [0.922879  , 0.07712109],
       [0.94723225, 0.05276779],
       [0.94199026, 0.05800974],
       [0.937316  , 0.06268404],
       [0.9409101 , 0.05908987],
       [0.92144674, 0.07855323],
       [0.9506474 , 0.0493526 ],
       [0.95444405, 0.04555598],
       [0.09292327, 0.90707666],
       [0.15136953, 0.8486305 ],
       [0.3080087 , 0.6919913 ],
       [0.14397359, 0.85602635],
       [0.14901783, 0.8509822 ],
       [0.18942367, 0.8105763 ],
       [0.23372948, 0.7662705 ],
       [0.19718605, 0.80281395],
       [0.15527081, 0.8447291 ],
       [0.14794283, 0.8520572 ],
       [0.